# 07. AED 공백 분석 — 위험도 × AED

## 이 노트북이 하는 일
위험격자(05·06) 위에 AED(축4) 106개를 겹쳐 **"위험한데 AED 없는 공백"** 을 찾고, AED가 **저지대에 편중**됐음을 표고 분포로 보인다.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 야간 기준인가:** 심정지는 밤에도 발생하는데 주간만 여는 AED는 무용. 야간 접근 가능한 AED만으로 커버를 다시 계산해야 실제 대응력이 보인다.
- **왜 100m 커버 반경인가:** 골든타임 내 도보 왕복 가능 거리 근사. 산복도로 경사를 반영 안 했으므로 실제 커버는 더 좁고, 공백은 오히려 과소추정(보수적).
- **왜 cKDTree인가:** 격자 415 × AED 106을 전부 거리계산하면 느림. KD트리로 최근접·반경내 탐색을 빠르게.
- **왜 표고를 그래프 노드에서 가져오나:** 06 그래프 노드에 이미 표고가 있어, AED·격자를 최근접 노드에 붙이면 추가 API 호출 없이 표고를 얻는다(오프라인·빠름).
- **왜 공백=상위30% 위험 & 야간 미커버인가:** '위험한 곳'과 'AED 없는 곳'의 교집합이 정책적으로 메워야 할 지점.

## 데이터 출처
- 위험격자: 06 / AED: 축4(15000652) / 표고: 06 그래프(SRTM 30m).

In [ ]:
%pip install matplotlib scipy

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd            # 수치·표
import geopandas as gpd, osmnx as ox        # 지리표·그래프
from scipy.spatial import cKDTree           # 빠른 최근접/반경 탐색
import folium, matplotlib                   # 지도·그래프
matplotlib.use("Agg")                       # 화면 없는 환경에서 그림 저장용 백엔드
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 히스토그램 한글 폰트(Windows)
plt.rcParams["axes.unicode_minus"] = False      # 음수 기호 깨짐 방지
CRS_WGS, CRS_M = 4326, 5186                 # 위경도 / 평면
R_COVER = 100                               # AED 커버 반경(m)
os.makedirs("outputs", exist_ok=True)

## 1. 데이터 로드 — 위험격자 · AED · 그래프(표고)

In [ ]:
grid = gpd.read_parquet("outputs/grid_risk_conn.parquet")                             # 06 위험격자(EPSG:5186)
print("위험격자:", len(grid), "| risk_norm 범위: %.2f~%.2f" % (grid["risk_norm"].min(), grid["risk_norm"].max()))

aed = pd.read_csv("outputs/aed_donggu.csv")                                           # 축4 AED 106개(좌표·운영시간)
aed = aed.dropna(subset=["wgs84Lat","wgs84Lon"]).copy()                              # 좌표 없는 행 제거
aed_g = gpd.GeoDataFrame(aed, geometry=gpd.points_from_xy(aed["wgs84Lon"], aed["wgs84Lat"]),
                         crs=CRS_WGS).to_crs(CRS_M)                                   # 점 도형 → 5186
print("AED:", len(aed_g))

def _sf(x):                                                                           # 빈 속성 안전 변환
    try: return float(x)
    except: return float("nan")
G = ox.load_graphml("outputs/graph_drive_conn.graphml", node_dtypes={"elev":_sf})     # 06 그래프(표고 있음)

## 2. AED 야간 접근성 분류
월요일 종료시각(monEndTme)이 24시(2400/0000) 또는 22시 이후면 '야간 접근 가능'으로 근사.

In [ ]:
def to_hhmm(v):                                                                       # 운영시간 값을 정수 시각으로 (CSV에서 2400.0 등으로 올 수 있음)
    try: return int(float(v))
    except: return None
def is_night(row):                                                                    # 야간 접근 가능 판정
    e = to_hhmm(row.get("monEndTme")); s = to_hhmm(row.get("monSttTme"))              # 종료·시작 시각
    if e is None: return False
    if s == 0 and e in (0, 2400, 2359): return True                                   # 0000~2400 = 24시간
    return e >= 2200                                                                  # 22시 이후까지 운영이면 야간 가능
aed_g["night"] = aed_g.apply(is_night, axis=1)                                         # 각 AED에 야간여부 부여
print("전체 AED:", len(aed_g), "| 야간 접근 가능:", int(aed_g["night"].sum()),
      "| 주간 전용:", int((~aed_g["night"]).sum()))

## 3. 격자별 최근접 AED 거리 + 커버 여부 (전체 / 야간)

In [ ]:
gc = np.c_[grid["cx"].values, grid["cy"].values]                                      # 격자 중심 좌표 배열
def nearest(mask):                                                                    # 주어진 AED 부분집합에 대해
    pts = np.c_[aed_g.loc[mask,"geometry"].x.values, aed_g.loc[mask,"geometry"].y.values]  # 그 AED 좌표
    if len(pts)==0: return np.full(len(grid), np.inf), np.zeros(len(grid),int)         # 없으면 무한대·0
    tree = cKDTree(pts)                                                               # KD트리 구축
    dist,_ = tree.query(gc, k=1)                                                      # 격자별 최근접 AED 거리
    cnt = tree.query_ball_point(gc, R_COVER)                                          # 반경 100m 내 AED 인덱스들
    return dist, np.array([len(c) for c in cnt])                                      # 거리 / 반경내 개수

grid["aed_dist_all"],   grid["aed_cnt_all"]   = nearest(aed_g.index)                  # 전체 AED 기준
grid["aed_dist_night"], grid["aed_cnt_night"] = nearest(aed_g["night"].values)        # 야간 AED 기준
grid["covered_all"]   = grid["aed_cnt_all"]   > 0                                     # 전체 커버 여부
grid["covered_night"] = grid["aed_cnt_night"] > 0                                     # 야간 커버 여부
print("커버(전체 AED, %dm내): %d/%d 격자" % (R_COVER, grid["covered_all"].sum(), len(grid)))
print("커버(야간 AED, %dm내): %d/%d 격자" % (R_COVER, grid["covered_night"].sum(), len(grid)))

## 4. 공백 격자 정의
공백 = 고위험(상위 30%) AND 야간 AED 커버 없음.

In [ ]:
thr = grid["risk_norm"].quantile(0.70)                                                # 위험도 상위 30% 경계값
grid["high_risk"] = grid["risk_norm"] >= thr                                          # 고위험 격자 표시
grid["gap"] = grid["high_risk"] & (~grid["covered_night"])                            # 공백 = 고위험 & 야간 미커버
n_high = int(grid["high_risk"].sum()); n_gap = int(grid["gap"].sum())
print(f"고위험 격자(상위30%, risk>={thr:.2f}): {n_high}")
print(f"그중 야간 AED 공백: {n_gap} ({n_gap/max(n_high,1)*100:.0f}%)")
print("\n공백 격자 소속 행정동:")
print(grid.loc[grid["gap"],"dong"].value_counts().to_string())

## 5. AED 표고 분포 — 저지대 편중 (핵심 논거)

In [ ]:
aed_nodes = ox.distance.nearest_nodes(G, X=aed_g.geometry.x.values, Y=aed_g.geometry.y.values)  # AED→최근접 노드
aed_g["elev"] = [G.nodes[n].get("elev", np.nan) for n in aed_nodes]                    # 그 노드의 표고를 AED 표고로
grid["elev"]  = [G.nodes[n].get("elev", np.nan) for n in grid["entry_node"]]           # 격자 표고 = 진입점 노드 표고

aed_el = pd.to_numeric(aed_g["elev"], errors="coerce").dropna()                        # AED 표고 분포
hi_el  = pd.to_numeric(grid.loc[grid["high_risk"],"elev"], errors="coerce").dropna()   # 고위험 격자 표고 분포
print("AED 표고        중앙 %.0fm (25~75%%: %.0f~%.0f)" % (aed_el.median(), aed_el.quantile(.25), aed_el.quantile(.75)))
print("고위험격자 표고 중앙 %.0fm (25~75%%: %.0f~%.0f)" % (hi_el.median(), hi_el.quantile(.25), hi_el.quantile(.75)))

fig, ax = plt.subplots(figsize=(7,4))                                                 # 히스토그램 그리기
ax.hist(aed_el, bins=20, alpha=0.6, label=f"AED (n={len(aed_el)})", color="#2c7fb8")   # AED(파랑): 저지대 밀집
ax.hist(hi_el, bins=20, alpha=0.6, label=f"고위험 격자 (n={len(hi_el)})", color="#d7191c")  # 위험(빨강): 고지대
ax.set_xlabel("표고 (m)"); ax.set_ylabel("빈도"); ax.legend()
ax.set_title("AED vs 고위험 격자 표고 분포")
fig.tight_layout(); fig.savefig("outputs/aed_elev_hist.png", dpi=120)                  # 그림 저장(발표용)
print("히스토그램 저장: outputs/aed_elev_hist.png")
plt.show()

## 6. 공백 지도 (folium)

In [ ]:
gw = grid.to_crs(CRS_WGS); aedw = aed_g.to_crs(CRS_WGS)                                # 지도용 위경도
def rcol(v):                                                                          # 위험도→색
    t=float(v); r=int(255*min(t+0.1,1)); g=int(200*(1-t)); b=int(60*(1-t)); return f"#{r:02x}{g:02x}{b:02x}"

m = folium.Map(location=[35.122,129.045], zoom_start=14, tiles="cartodbpositron")
fg_risk = folium.FeatureGroup(name="위험도").add_to(m)                                 # 레이어1: 위험도 격자
for _,r in gw.iterrows():
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x,c=rcol(r["risk_norm"]):{"color":c,"weight":0.2,"fillColor":c,"fillOpacity":0.45}).add_to(fg_risk)
fg_gap = folium.FeatureGroup(name="⚠ 공백 격자(고위험+야간AED없음)").add_to(m)          # 레이어2: 공백(검은 테두리)
for _,r in gw[gw["gap"]].iterrows():
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x:{"color":"#000","weight":2,"fillColor":"#000","fillOpacity":0.0}).add_to(fg_gap)
fg_aed = folium.FeatureGroup(name="AED").add_to(m)                                     # 레이어3: AED(초록=야간, 회색=주간)
for _,a in aedw.iterrows():
    c = "#1a9641" if a["night"] else "#999999"
    folium.CircleMarker([a.geometry.y,a.geometry.x], radius=3, color=c, fill=True, fillOpacity=0.9,
        tooltip=f'{a.get("org","")} | {"야간O" if a["night"] else "주간"}').add_to(fg_aed)
legend=('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
        'border:1px solid #999;font-size:12px"><b>위험 × AED 공백</b><br>'
        '<span style="color:#d7191c">&#9644;</span> 고위험 격자 &nbsp; '
        '<span style="color:#000">▢</span> 공백(야간AED 없음)<br>'
        '<span style="color:#1a9641">●</span> 야간 AED &nbsp; <span style="color:#999">●</span> 주간 AED</div>')
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl().add_to(m)
m.save("outputs/gap_map_A4.html")                                                      # 공백 지도 저장
print("지도 저장: outputs/gap_map_A4.html")
m

## 7. 저장

In [ ]:
keep=["grid_id","cx","cy","dong","risk_norm","high_risk",                             # 저장할 컬럼
      "aed_dist_all","aed_dist_night","covered_all","covered_night","gap","elev","geometry"]
out=grid[keep].copy()
out.to_parquet("outputs/grid_gap_A4.parquet")                                          # 공백 분석 격자 저장(08이 읽음)
try: out.to_file("outputs/grid_gap_A4.gpkg", driver="GPKG")
except Exception as e: print("gpkg 경고:", e)
aed_g.drop(columns="geometry").assign(lon=aed_g.geometry.to_crs(4326).x, lat=aed_g.geometry.to_crs(4326).y)\
     .to_csv("outputs/aed_donggu_classified.csv", index=False, encoding="utf-8-sig")   # AED 야간분류 저장
print("저장:", [f for f in sorted(os.listdir("outputs")) if "gap" in f or "classified" in f or "elev" in f])